# Train Notebook

### 1. Import Libraries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib

### 2. Data Loading and Preprocessing

In [2]:
# Load the dataset
try:
    df = pd.read_excel('rccarbonation.xlsx')
    
    # Mapping Dictionary (Text -> Number)
    mapping_dict = {
        r'\bPIA\b': '0', r'\bUEA\b': '1', r'\bPEA\b': '2',
        r'\bCPII Z\b': '0', r'\bCPV - ARI\b': '1', r'\bCPIV\b': '2',
        r'\bCPII F\b': '3', r'\bCPIII\b': '4', r'\bCPII E\b': '5',
    }
    
    # Apply replacement
    df.replace(mapping_dict, regex=True, inplace=True)
    
    # Ensure columns are numeric
    cols_to_numeric = ['Exposure conditions', 'Type of cement']
    for col in cols_to_numeric:
        df[col] = pd.to_numeric(df[col])

    print("Data loaded and processed. Shape:", df.shape)
    display(df.head())

except FileNotFoundError:
    print("ERROR: The file 'rccarbonation.xlsx' was not found.")

Data loaded and processed. Shape: (20000, 7)


,CCO₂ (%),fc (MPa),RH (%),Type of cement,Exposure conditions,t (years),Possan
0,0.08,25.9,35.4,0,0,9,10.1384
1,0.07,21.7,43.8,1,1,92,23.2231
2,0.08,30.3,83.7,2,1,58,13.2715
3,0.15,22.8,37.6,2,0,42,45.1498
4,0.03,26.4,65.2,3,1,96,22.6178


# 3. Model Training and Evaluation

In this section, we define the features ($X$) and the target variable ($y$).
The dataset is split into training (80%) and testing (20%) sets to ensure a robust validation.

We utilize the **HistGradientBoostingRegressor**, a widely used algorithm for regression tasks, and evaluate its performance using the Coefficient of Determination ($R^2$).

In [3]:
# Separate Features (X) and Target (y)
X = df.drop('Possan', axis=1)
y = df['Possan']

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Instantiate and Train the Model
print("Starting training...")
model = HistGradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)

# Evaluation
score = model.score(X_test, y_test)
print(f"Training completed! R² on test set: {score:.4f}")

Starting training...
Training completed! R² on test set: 0.9922


### 3.1. Detailed Performance Evaluation ($R^2$ vs Adjusted $R^2$)

While the standard Coefficient of Determination ($R^2$) indicates how well the model fits the data, it can be biased if the number of features is large.
To address this, we calculate the **Adjusted $R^2$**, which adds a penalty for model complexity (number of predictors).

Since `scikit-learn` does not provide a direct function for Adjusted $R^2$, we compute it manually using the formula:
$$R^2_{adj} = 1 - (1 - R^2) \frac{n - 1}{n - p - 1}$$
Where:
* $n$ is the number of samples.
* $p$ is the number of features (predictors).

In [4]:
from sklearn.metrics import r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate Standard R²
r2 = r2_score(y_test, y_pred)

# Calculate Adjusted R² manually
n = len(y_test)      
p = X_test.shape[1]   

r2_adjusted = 1 - (1 - r2) * (n - 1) / (n - p - 1)

# Display Results
print(f"Model Performance Metrics")
print(f"R² (Standard):   {r2:.4f}")
print(f"R² Adjusted:     {r2_adjusted:.4f}")

# Check difference (if minimal, the model is not overfitting due to excess features)
diff = r2 - r2_adjusted
print(f"Difference:      {diff:.5f}")

Model Performance Metrics
R² (Standard):   0.9922
R² Adjusted:     0.9922
Difference:      0.00001


# 4. Save the model

In [5]:
joblib.dump(model, 'modelo_carbonatacao.pkl')
joblib.dump(X_train.columns, 'colunas_modelo.pkl')

print("Model and columns saved as .pkl files!")

Model and columns saved as .pkl files!
